# 24 · Controlling tools: tool_choice and strict schemas
Level L2 · Part 3 · professional · ~25 min · builds on 22

## Goal
Force, forbid or free Scout's tool use with **tool_choice**, and use a **strict tool schema** so the
arguments always match: you will force Scout to call `search_notes` first and catch a malformed argument.

## The idea
In lesson 22 the model decided on its own whether to call `search_notes`. That is the default, `auto`.
Two request settings take some of that freedom away. **tool_choice** says *whether* a tool must be
called: `auto` (model decides), `any` / `required` (must call some tool), a named tool (must call this
one) or `none` (no tools). **Strict** mode says *how* the arguments look: the provider constrains the
model's output so the arguments always match your JSON Schema. Think of a bank form. tool_choice is the
clerk saying "fill in form A before anything else"; strict is a form whose boxes only accept digits.

## Picture
```mermaid
flowchart LR
    Q[question + tools] --> C{tool_choice}
    C -- auto --> E[text or tool call]
    C -- "any / named tool" --> F[tool call]
    C -- none --> T[text only]
    F --> S{strict?}
    S -- yes --> OK[args match schema]
    S -- no --> V[args may be malformed: validate]
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 22. Without a key, the Gemini calls replay from
`../data/llm_cassettes/24_tool-choice-and-strict.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("24_tool-choice-and-strict")

LLM mode: replay (24_tool-choice-and-strict.yaml)


Setup from lesson 22, unchanged: the notes, the client and model, `search_notes`, its schema and the
Gemini declaration.

In [2]:
import json
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

Lesson 22's tool schema, unchanged.

In [3]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}. Use it before answering any question "
                   "about what the notes say.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}},
        "required": ["query"],
    },
}
TOOLS = {"search_notes": search_notes}
print("model:", MODEL, "| tools:", list(TOOLS))

model: gemini-3.5-flash-lite | tools: ['search_notes']


### 2. A fake model that obeys tool_choice
Our scripted model behaves like a real one in `auto` mode: it calls the tool only when the question
mentions "notes". Otherwise it answers from "memory", which for Scout is a bug: the answer is not
grounded in the user's notes. `drafts` are the arguments it would like to send, best first. We use
`strict` in step 4.

In [4]:
class FakeScout:
    def __init__(self, drafts: list[dict], strict=None):
        self.drafts, self.strict, self.calls = drafts, strict, 0

    def pick_args(self) -> dict:
        if self.strict is None:                       # free decoding: the favourite draft, as it is
            return self.drafts[0]
        for draft in self.drafts:                     # constrained: only schema-valid drafts survive
            try:
                self.strict.model_validate(draft); return draft
            except ValueError:
                continue
        raise RuntimeError("no draft fits the schema")

    def __call__(self, messages: list[dict], tools: list[dict], tool_choice="auto") -> dict:
        last = messages[-1]
        if last["role"] == "tool" and tool_choice == "auto":
            return {"role": "assistant", "text": f"From your notes: {last['content']}"}
        wants_tool = last["role"] == "user" and "notes" in last["text"].lower()
        if tool_choice == "none" or (tool_choice == "auto" and not wants_tool):
            return {"role": "assistant", "text": "Mars has two moons, I think."}   # from memory
        name = tool_choice["name"] if isinstance(tool_choice, dict) else tools[0]["name"]
        self.calls += 1
        return {"role": "assistant", "tool_calls": [
            {"id": f"call_{self.calls}", "name": name, "args": self.pick_args()}]}

Same question, four tool_choice values. The fake uses the vendor-neutral spelling: `"auto"`,
`"required"`, `"none"`, or `{"name": ...}` for one named tool.

In [5]:
QUESTION = "How many moons does Mars have?"
fake = FakeScout(drafts=[{"query": "Mars"}])
for choice in ["auto", "required", {"name": "search_notes"}, "none"]:
    reply = fake([{"role": "user", "text": QUESTION}], tools=[SEARCH_NOTES_SCHEMA], tool_choice=choice)
    print(f"{str(choice):26} ->", reply.get("text") or reply["tool_calls"][0])

auto                       -> Mars has two moons, I think.
required                   -> {'id': 'call_1', 'name': 'search_notes', 'args': {'query': 'Mars'}}
{'name': 'search_notes'}   -> {'id': 'call_2', 'name': 'search_notes', 'args': {'query': 'Mars'}}
none                       -> Mars has two moons, I think.


With `auto` the model skipped the notes and guessed. `required` and the named tool force the search.
`none` keeps the tools in the request but forbids calling them: useful for a final "now just answer"
turn.

### 3. A malformed argument
Now give the fake the habits real models have without strict mode: an extra field the function does not
take (holding a number written as a string). That draft comes first, so the fake sends it.

In [6]:
SLOPPY = [{"query": "Mars", "max_results": "3"}, {"query": "Mars"}]
loose = FakeScout(drafts=SLOPPY)
call = loose([{"role": "user", "text": QUESTION}], tools=[SEARCH_NOTES_SCHEMA],
             tool_choice={"name": "search_notes"})["tool_calls"][0]
print("tool call args:", call["args"])

tool call args: {'query': 'Mars', 'max_results': '3'}


The schema in lesson 22 never said "no other fields", so nothing stopped `max_results`. Write the
arguments as a Pydantic model (lesson 20) with `extra="forbid"` and `strict=True`, and check before
running anything.

In [7]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class SearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    query: str = Field(description="one word to look for, e.g. 'Mars'")

try:
    SearchArgs.model_validate(call["args"])
except ValidationError as e:
    print("caught:", e.errors()[0]["type"], "on", e.errors()[0]["loc"])

caught: extra_forbidden on ('max_results',)


Caught before `search_notes` ran. This check is your last line of defence. Strict mode moves the same
rule to the provider, so the bad draft is never produced.

### 4. Strict schemas
Providers accept strict mode only for schemas that close every door. The two common rules (Anthropic and
OpenAI): every object has `"additionalProperties": false`, and (OpenAI) every property is in `required`.
A small checker:

In [8]:
def strict_problems(schema: dict, path="$") -> list[str]:
    problems = []
    if schema.get("type") == "object":
        if schema.get("additionalProperties") is not False:
            problems.append(f"{path}: additionalProperties is not false")
        missing = set(schema.get("properties", {})) - set(schema.get("required", []))
        problems += [f"{path}.{name}: not in required" for name in sorted(missing)]
        for name, sub in schema.get("properties", {}).items():
            problems += strict_problems(sub, f"{path}.{name}")
    return problems

STRICT_SCHEMA = SearchArgs.model_json_schema()
print("lesson 22 schema:", strict_problems(SEARCH_NOTES_SCHEMA["parameters"]))
print("SearchArgs schema:", strict_problems(STRICT_SCHEMA))
print(json.dumps(STRICT_SCHEMA))

lesson 22 schema: ['$: additionalProperties is not false']
SearchArgs schema: []
{"additionalProperties": false, "properties": {"query": {"description": "one word to look for, e.g. 'Mars'", "title": "Query", "type": "string"}}, "required": ["query"], "title": "SearchArgs", "type": "object"}


The Pydantic model gives a strict-ready schema for free. A real provider compiles it into a grammar and
only samples tokens that fit. Our fake imitates that: with `strict=SearchArgs` it skips drafts the
schema rejects.

In [9]:
strict_fake = FakeScout(drafts=SLOPPY, strict=SearchArgs)
call = strict_fake([{"role": "user", "text": QUESTION}], tools=[SEARCH_NOTES_SCHEMA],
                   tool_choice={"name": "search_notes"})["tool_calls"][0]
print("tool call args:", call["args"])
print("run it:", [n["id"] for n in TOOLS[call["name"]](**SearchArgs.model_validate(call["args"]).model_dump())])

tool call args: {'query': 'Mars'}
run it: [1, 3, 5]


### 5. Gemini: auto, forced and none
Gemini puts tool_choice in `tool_config.function_calling_config`. Its modes are `AUTO`, `ANY`, `NONE`
and `VALIDATED`; `allowed_function_names` narrows `ANY` to named tools. A helper builds the config.

In [10]:
search_decl = types.FunctionDeclaration(
    name=SEARCH_NOTES_SCHEMA["name"], description=SEARCH_NOTES_SCHEMA["description"],
    parameters_json_schema=SEARCH_NOTES_SCHEMA["parameters"])

def tool_cfg(mode: str, names=None, decl=search_decl) -> types.GenerateContentConfig:
    fcc = types.FunctionCallingConfig(mode=mode, allowed_function_names=names)
    return types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[decl])], automatic_function_calling=NO_AFC,
        tool_config=types.ToolConfig(function_calling_config=fcc), max_output_tokens=300)

contents = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]
print(tool_cfg("ANY", ["search_notes"]).tool_config.function_calling_config)

allowed_function_names=['search_notes'] mode=<FunctionCallingConfigMode.ANY: 'ANY'> stream_function_call_arguments=None


The same question in all three modes. We print the tool call if there is one, else the text.

In [11]:
replies = {}
for mode, names in [("AUTO", None), ("ANY", ["search_notes"]), ("NONE", None)]:
    replies[mode] = client.models.generate_content(model=MODEL, contents=contents,
                                                   config=tool_cfg(mode, names))
    fcs = replies[mode].function_calls
    shown = f"tool call {fcs[0].name}({dict(fcs[0].args)})" if fcs else repr(replies[mode].text)
    print(f"{mode:4} -> {shown}")

AUTO -> 'Mars has two moons, named Phobos and Deimos.'
ANY  -> tool call search_notes({'query': 'Mars'})
NONE -> 'Mars has **2** moons. Their names are **Phobos** and **Deimos**.'


In the recorded run, `AUTO` answered "two moons, Phobos and Deimos" from Gemini's own knowledge: right
here, but not from the user's notes, which is what Scout promises. `ANY` with
`allowed_function_names=["search_notes"]` produced `search_notes(query="Mars")`. `NONE` answered in text
although the tool was declared.

### 6. Turn forcing off after the first call
Forcing is for the *first* step. Run the forced search, send the result back, and look at what happens
if `ANY` stays on compared with switching back to `AUTO`.

In [12]:
fc = replies["ANY"].function_calls[0]
hits = TOOLS[fc.name](**fc.args)
history = contents + [replies["ANY"].candidates[0].content, types.Content(role="user", parts=[
    types.Part(function_response=types.FunctionResponse(id=fc.id, name=fc.name, response={"result": hits}))])]
for mode, names in [("ANY", ["search_notes"]), ("AUTO", None)]:
    r = client.models.generate_content(model=MODEL, contents=history, config=tool_cfg(mode, names))
    shown = f"tool call {r.function_calls[0].name}({dict(r.function_calls[0].args)})" if r.function_calls else repr(r.text)
    print(f"call 2 with {mode:4} -> {shown}")

call 2 with ANY  -> tool call search_notes({'query': 'moons'})
call 2 with AUTO -> 'Based on your notes, Mars has two small moons: Phobos and Deimos.'


With `ANY` still on, Gemini could not answer: it asked for another search (`query="moons"`). With `AUTO`
it answered from the result: "Based on your notes, Mars has two small moons". Force step one, free the
rest.

### 7. Gemini's strict mode: VALIDATED
Gemini has no `strict` flag on the declaration. Its `VALIDATED` mode "ensures function schema adherence"
(Gemini docs) while still letting the model answer in text. We send the strict Pydantic schema and still
run our own check, because a guarantee from the provider is not a check in your code.

In [13]:
strict_decl = types.FunctionDeclaration(name="search_notes", description=SEARCH_NOTES_SCHEMA["description"],
                                        parameters_json_schema=STRICT_SCHEMA)
ask = [types.Content(role="user", parts=[types.Part(text="Which of my notes mention Mars?")])]
r = client.models.generate_content(model=MODEL, contents=ask, config=tool_cfg("VALIDATED", decl=strict_decl))
for fc in r.function_calls or []:
    args = SearchArgs.model_validate(fc.args)
    print("validated args:", args, "->", [n["id"] for n in search_notes(**args.model_dump())])
print("text:", r.text if not r.function_calls else None)

validated args: query='Mars' -> [1, 3, 5]
text: None


Gemini chose the tool with `query="Mars"`, the arguments passed our Pydantic check and the search found
notes 1, 3 and 5. No text this turn, because it called the tool.

### 8. The same two knobs at three vendors
Built by hand, as in lesson 23 (no call is sent). "Force `search_notes`, strict arguments":

In [14]:
PAYLOADS = {
    "Anthropic Messages": {
        "tools": [{"name": "search_notes", "strict": True, "input_schema": STRICT_SCHEMA}],
        "tool_choice": {"type": "tool", "name": "search_notes"}},
    "OpenAI Responses": {
        "tools": [{"type": "function", "name": "search_notes", "strict": True, "parameters": STRICT_SCHEMA}],
        "tool_choice": {"type": "function", "name": "search_notes"}},
    "Gemini generate_content": {
        "tool_config": {"function_calling_config": {"mode": "ANY", "allowed_function_names": ["search_notes"]}}},
}
for vendor, body in PAYLOADS.items():
    print(f"{vendor:24}", {k: v for k, v in body.items() if k != "tools"})

Anthropic Messages       {'tool_choice': {'type': 'tool', 'name': 'search_notes'}}
OpenAI Responses         {'tool_choice': {'type': 'function', 'name': 'search_notes'}}
Gemini generate_content  {'tool_config': {'function_calling_config': {'mode': 'ANY', 'allowed_function_names': ['search_notes']}}}


| You want | Anthropic `tool_choice` | OpenAI `tool_choice` | Gemini `function_calling_config.mode` |
|---|---|---|---|
| model decides (default) | `{"type": "auto"}` | `"auto"` | `AUTO` |
| must call some tool | `{"type": "any"}` | `"required"` | `ANY` |
| must call this tool | `{"type": "tool", "name": ...}` | `{"type": "function", "name": ...}` | `ANY` + `allowed_function_names` |
| no tool calls | `{"type": "none"}` | `"none"` | `NONE` |
| strict arguments | `"strict": true` per tool | `"strict": true` per tool | `VALIDATED` mode |

## At work
- **Forced tool use is not available everywhere.** Anthropic lists Claude Opus 5.5 and Sonnet 5.5 as
  returning 400 for `any` / `tool` (manual extended thinking too); its advice is `auto` + strict tools +
  a clear prompt. Keep tool_choice in config. → lesson 33 (routing without forcing)
- **Strict schemas are compiled and cached.** Anthropic turns a strict schema into a grammar cached up
  to 24 h: keep personal health data out of property names and enums; max 20 strict tools per request.
  → lesson 50
- **Teams still validate arguments.** Strict guarantees the *shape*, not that `"../../etc"` is a safe
  file name. Validate and return failures to the model. → lesson 28, lesson 51
- **Changing tool_choice costs cache.** With Anthropic prompt caching, a new `tool_choice` invalidates
  cached messages, so flipping it every turn costs more. → lesson 54

## What just happened
- **tool_choice** decided *whether* Scout calls a tool: `auto` let it guess from memory, a **forced tool
  call** (`required`, a named tool, Gemini `ANY` + `allowed_function_names`) made it search first,
  `none` forbade tools.
- Forcing is for one step: after the first call we switched back to `AUTO` so the model could answer.
- A sloppy argument (`max_results: "3"`) was caught by a Pydantic model with `extra="forbid"`.
- A **strict tool schema** (`additionalProperties: false`, every field required) lets the provider
  generate only valid arguments; Gemini's version is the `VALIDATED` mode.

## Common mistakes
**1. Running the arguments without checking them.** Without strict mode the sloppy draft goes straight
into the function, and Python fails on the extra field (a real tool might instead do something wrong).

In [15]:
bad = loose([{"role": "user", "text": QUESTION}], tools=[SEARCH_NOTES_SCHEMA], tool_choice="required")
c = bad["tool_calls"][0]
TOOLS[c["name"]](**c["args"])

TypeError: search_notes() got an unexpected keyword argument 'max_results'

Fix: validate first, and send a clear error back to the model if it fails (lesson 28 does that in full).

In [16]:
try:
    print(TOOLS[c["name"]](**SearchArgs.model_validate(c["args"]).model_dump()))
except ValidationError as e:
    print("send back to the model:", e.errors()[0]["msg"], e.errors()[0]["loc"])

send back to the model: Extra inputs are not permitted ('max_results',)


**2. Leaving the tool forced for the whole conversation.** The model can never answer: every reply is
another tool call, so code that expects text breaks (or a loop runs until its step limit).

In [17]:
msgs = [{"role": "user", "text": QUESTION}]
msgs.append(strict_fake(msgs, tools=[SEARCH_NOTES_SCHEMA], tool_choice="required"))
tc = msgs[-1]["tool_calls"][0]
msgs.append({"role": "tool", "call_id": tc["id"], "content": json.dumps([n["id"] for n in search_notes("Mars")])})
print(strict_fake(msgs, tools=[SEARCH_NOTES_SCHEMA], tool_choice="required")["text"])

KeyError: 'text'

Fix: force only the first call, then go back to `auto`.

In [18]:
print(strict_fake(msgs, tools=[SEARCH_NOTES_SCHEMA], tool_choice="auto")["text"])

From your notes: [1, 3, 5]


## Try it
Scout wants an optional `limit` on `search_notes`. Write `SearchArgsV2` with `query: str` and
`limit: int | None` (required but nullable, so it passes `strict_problems`), check its schema, then run
`FakeScout(drafts=[{"query": "Mars", "limit": "2"}, {"query": "Mars", "limit": 2}], strict=SearchArgsV2)`
with tool_choice `"required"` and print the arguments it chose.

In [19]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [20]:
llm_replay.stop()

## Key terms
- **tool_choice** — the request setting that says whether the model may, must or must not call a tool: auto, any/required, a named tool, or none (Gemini: `function_calling_config.mode`).
- **forced tool call** — a tool call the model had to make because tool_choice was any/required or named the tool; the model cannot answer in text on that turn.
- **strict tool schema** — a tool schema the provider enforces while generating, so the arguments always match it; needs `additionalProperties: false` and (OpenAI) every field in `required`.
- **`VALIDATED` mode** — Gemini's function-calling mode that keeps tool arguments to the declared schema while still allowing a text answer.